# CP3 3.4 - Pemeriksaan dan Analisis Hasil Awal

Notebook ini memenuhi bagian **3.4**: membandingkan performa baseline, menentukan yang terkuat dan terlemah, mencatat temuan/anomali, dan menyiapkan insight untuk tahap berikutnya.

Sumber angka adalah artefak yang sudah dihasilkan Task 2 dan Task 3, bukan perhitungan ulang:

- `outputs/task2/baseline_comparison.csv`, `baseline_metrics_per_user.csv`, `baseline_top10.csv`
- `outputs/task3/task3_model_comparison.csv`, `task3_summary.json`, `task3_weighted_score_results.csv`, `task3_rrf_results.csv`

Tidak ada model yang dilatih ulang dan data test tidak dipakai.


## 1. Setup dan audit artefak

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "task4_initial_analysis.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Jalankan notebook dari folder proyek yang berisi src/task4_initial_analysis.py.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.task4_initial_analysis import (
    load_artifacts, audit_artifacts, summarize_models, identify_extremes,
    user_overlap, pairwise_wins, detect_anomalies, build_insights, export_analysis,
)

pd.set_option("display.width", 160)
print("Project root:", PROJECT_ROOT)


Project root: D:\Semester 7\Sisrek\Praktikum\proyek


Audit memastikan ketiga baseline dibandingkan pada protokol yang sama: jumlah pengguna identik, metrik berada di [0, 1], dan nilai Popularity konsisten di ketiga artefak. Jika salah satu syarat gagal, notebook berhenti.


In [2]:
artifacts = load_artifacts(PROJECT_ROOT)
audit = audit_artifacts(artifacts)
print("Pengguna evaluasi         :", f"{audit['users']:,}")
print("Baseline                  :", ", ".join(audit["baseline_models"]))
print("Metrik                    : Recall@%d dan NDCG@%d" % (audit["protocol_k"], audit["protocol_k"]))
print("Ambang relevan            : rating >= %s" % audit["protocol_positive_threshold"])
print("Data test dipakai         :", "ya" if audit["uses_test_data"] else "tidak")
print("Nilai Popularity (3 sumber):", [round(v, 12) for v in audit["popularity_recall_sources"]])


Pengguna evaluasi         : 11,659
Baseline                  : Item-CF, Popularity, TF-IDF CBF
Metrik                    : Recall@10 dan NDCG@10
Ambang relevan            : rating >= 4.0
Data test dipakai         : tidak
Nilai Popularity (3 sumber): [0.013498503733, 0.013498503733, 0.013498503733]


## 2. Perbandingan performa antar baseline

Baris baseline dihitung dari metrik per pengguna; baris fusi diambil dari ringkasan Task 3 karena hanya nilai agregatnya yang tersimpan. Kolom terakhir membandingkan Recall@10 terhadap Popularity.


In [3]:
table = summarize_models(artifacts)
display(
    table.assign(**{
        "recall@10": table["recall@10"].round(6),
        "ndcg@10": table["ndcg@10"].round(6),
        "hit_rate_pct": table["hit_rate_pct"].round(2),
        "mean_targets": table["mean_targets"].round(3),
        "recall_vs_popularity_pct": table["recall_vs_popularity_pct"].round(2),
    })
)


,model,sumber_per_user,users,recall@10,ndcg@10,users_with_hit,hit_rate_pct,mean_targets,unique_items_top10,unique_items_rank1,recall_vs_popularity_pct
0,RRF (k=100),False,11659,0.014198,0.009086,NaN,NaN,NaN,NaN,NaN,5.19
1,Popularity,True,11659,0.013499,0.009222,362.0,3.10,2.076,18.0,6.0,0.00
2,Weighted Score (CF=0.3),False,11659,0.012820,0.007930,NaN,NaN,NaN,NaN,NaN,-5.02
3,TF-IDF CBF,True,11659,0.012234,0.007492,248.0,2.13,2.076,16838.0,5108.0,-9.37
4,Item-CF,True,11659,0.005885,0.003351,128.0,1.10,2.076,18121.0,5076.0,-56.40


## 3. Baseline terkuat dan terlemah

In [4]:
extremes = identify_extremes(table)
for key, value in extremes.items():
    print(f"{key:32s}: {value}")


terkuat_recall                  : Popularity
terkuat_ndcg                    : Popularity
terlemah_recall                 : Item-CF
terlemah_ndcg                   : Item-CF
rasio_terlemah_terkuat_recall   : 0.43599525018214713


Terkuat pada Recall@10 dan NDCG@10 tidak harus model yang sama. Popularity adalah baseline tanpa personalisasi, sehingga posisinya yang tinggi sekaligus menandakan sinyal personal pada data ini lemah.


## 4. Irisan pengguna dan perbandingan berpasangan

In [5]:
overlap = user_overlap(artifacts["per_user"])
wins = pairwise_wins(artifacts["per_user"])
display(overlap)
display(wins)


,kombinasi_model_dengan_hit,n_users
0,tidak ada hit,10984
1,Popularity,333
2,TF-IDF CBF,207
3,Item-CF,73
4,Item-CF + TF-IDF CBF,33
5,Item-CF + Popularity,21
6,Popularity + TF-IDF CBF,7
7,Item-CF + Popularity + TF-IDF CBF,1


,model_a,model_b,a_lebih_baik,b_lebih_baik,seri
0,Item-CF,Popularity,106,340,11213
1,Item-CF,TF-IDF CBF,95,216,11348
2,Popularity,TF-IDF CBF,354,241,11064


## 5. Temuan dan anomali

In [6]:
anomalies = detect_anomalies(artifacts, table)
if not anomalies:
    print("Tidak ada anomali terdeteksi.")
for index, item in enumerate(anomalies, start=1):
    print(f"{index}. {item['judul']}")
    print(f"   {item['detail']}")
    print(f"   Dampak: {item['dampak']}")
    print()


1. Nilai Item-CF berbeda antara artefak Task 2 dan Task 3
   recall@10 = 0.0058853 di baseline_comparison.csv vs 0.0057995 di task3_model_comparison.csv (selisih 8.58e-05, 1.46% relatif); ndcg@10 = 0.0033510 di baseline_comparison.csv vs 0.0033230 di task3_model_comparison.csv (selisih 2.79e-05, 0.83% relatif). Kemungkinan cache rekomendasi dipakai ulang setelah kode berubah; hapus outputs/task3/cache lalu jalankan ulang sebelum angka dikutip.
   Dampak: angka CF/CBF belum bisa dikutip sebagai satu nilai final

2. Nilai TF-IDF CBF berbeda antara artefak Task 2 dan Task 3
   recall@10 = 0.0122340 di baseline_comparison.csv vs 0.0121482 di task3_model_comparison.csv (selisih 8.58e-05, 0.70% relatif); ndcg@10 = 0.0074921 di baseline_comparison.csv vs 0.0074909 di task3_model_comparison.csv (selisih 1.20e-06, 0.02% relatif). Kemungkinan cache rekomendasi dipakai ulang setelah kode berubah; hapus outputs/task3/cache lalu jalankan ulang sebelum angka dikutip.
   Dampak: angka CF/CBF belum bi

## 6. Insight untuk tahap berikutnya

In [7]:
insights = build_insights(table, overlap, anomalies)
for index, text in enumerate(insights, start=1):
    print(f"{index}. {text}")


1. Item-CF adalah baseline terlemah: Recall@10 0.005885, sekitar 56% di bawah Popularity, dan hanya 128 pengguna mendapat satu pun item relevan. Fallback popularitas di recommend_from_model membuat Top-10 Item-CF sebagian besar bukan hasil kemiripan item, jadi kontribusi CF murni perlu diukur terpisah.
2. Ketiga baseline hampir tidak beririsan: hanya 1 pengguna mendapat hit di ketiganya, sedangkan 333 pengguna hanya tertolong Popularity, 207 hanya TF-IDF CBF, dan 73 hanya Item-CF. Ini menjelaskan mengapa fusi berbasis ranking bisa menaikkan Recall di atas semua komponennya.
3. Prioritas perbaikan berikutnya adalah memperbaiki skor Item-CF (normalisasi jumlah tetangga, ambang co-occurrence minimum, dan memisahkan fallback popularitas sebagai varian tersendiri) sebelum menambah model baru seperti MF.
4. Popularity sebaiknya ikut dijadikan komponen fusi, bukan hanya pembanding: karena ia menang pada pengguna yang tidak tertolong CF/CBF, daftar ketiga berpeluang memperbaiki NDCG fusi yang 

## 7. Simpan hasil

In [8]:
paths = export_analysis(
    PROJECT_ROOT, artifacts, audit, table, extremes, overlap, wins, anomalies, insights
)
for path in paths:
    print("-", path.relative_to(PROJECT_ROOT).as_posix())


- outputs/task4/task4_model_comparison.csv
- outputs/task4/task4_user_overlap.csv
- outputs/task4/task4_pairwise_wins.csv
- outputs/task4/task4_initial_analysis.json
- outputs/task4/task4_analysis_summary.md
